#  Training YOLO on Fire Detection for Satellite Scenario

## Overview
In this notebook, we train a **YOLOv8** model on a custom dataset designed for detecting **fire scenarios** in **satellite images**. This dataset represents one of the specific scenarios for our **Mixture of Experts (MoE)** model, where each expert specializes in a different scenario (e.g., fire detection in indoor, indoor, satellite, or far-field environments).

### Key Steps:
- **Dataset**: The model is trained using a **satellite fire detection dataset**, which consists of images with fire-related features captured from satellite imagery.
- **YOLOv8 Training**: The YOLOv8 model is fine-tuned on this dataset, learning to identify fire-related objects.
- **Scenario Expert**: This trained model acts as an expert specifically for detecting fire in satellite images and is part of a broader MoE-based approach for multi-scenario detection.



##  Data Loading & Preprocessing


In [1]:
import os
import random
import shutil

# Set seed for reproducibility
random.seed(42)

# Paths
original_images_dir = "train/images"
original_labels_dir = "train/labels"

train_subset_images_dir = "satellite_fire_dataset/images/train_subset"
train_subset_labels_dir = "satellite_fire_dataset/labels/train_subset"

test_subset_images_dir = "satellite_fire_dataset/images/test_subset"
test_subset_labels_dir = "satellite_fire_dataset/labels/test_subset"

# Create directories if they don't exist
os.makedirs(train_subset_images_dir, exist_ok=True)
os.makedirs(train_subset_labels_dir, exist_ok=True)
os.makedirs(test_subset_images_dir, exist_ok=True)
os.makedirs(test_subset_labels_dir, exist_ok=True)

# List all image files
all_images = [f for f in os.listdir(original_images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# Randomly select 3100 images total
sampled_images = random.sample(all_images, 3100)

# Split into 3000 train, 100 test
train_images = sampled_images[:3000]
test_images = sampled_images[3000:]

# Helper function to copy images and labels
def copy_data(img_list, img_dest, label_dest):
    for img_file in img_list:
        # Copy image
        shutil.copy(
            os.path.join(original_images_dir, img_file),
            os.path.join(img_dest, img_file)
        )

        # Get corresponding label file
        label_file = os.path.splitext(img_file)[0] + ".txt"
        original_label_path = os.path.join(original_labels_dir, label_file)

        # Copy label if it exists
        if os.path.exists(original_label_path):
            shutil.copy(original_label_path, os.path.join(label_dest, label_file))

# Copy training data
copy_data(train_images, train_subset_images_dir, train_subset_labels_dir)

# Copy test data
copy_data(test_images, test_subset_images_dir, test_subset_labels_dir)

print("✅ Sampled 3000 training and 100 test images and labels from original training set.")


✅ Sampled 3000 training and 100 test images and labels from original training set.


##  Training YOLOv8 on Satellite Fire Detection

We utilize the **YOLOv8** object detection model to train on a custom dataset defined in `satellite.yaml`. The configuration for training is as follows:

- 🧠 **Model**: `yolov8n.pt` (Nano variant — optimized for speed and prototyping)
- 📁 **Dataset**: Defined in `satellite.yaml`
- 🖼️ **Image Size**: 640×640
- 🔁 **Epochs**: 100
- 📦 **Batch Size**: 16
- 💻 **Device**: GPU (device 0)




In [6]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Current device:", torch.cuda.get_device_name(0))

GPU available: True
Current device: NVIDIA GeForce RTX 3070


In [ ]:
# ---------- Imports ----------
import os
import cv2
import torch
import numpy as np
import albumentations as A
from tqdm import tqdm
import torch.nn as nn
import matplotlib.pyplot as plt
from timm import create_model
from sklearn.metrics import precision_score, recall_score, f1_score
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import box_iou, MultiScaleRoIAlign
from torchvision.models.detection.image_list import ImageList

# ---------- Dataset ----------
class FireDetectionDataset(Dataset):
    def __init__(self, images_dir, labels_dir, transform=None):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_files = [f for f in os.listdir(images_dir) if f.endswith(('.jpg', '.png'))]
        self.transform = transform

    def __getitem__(self, idx):
        img_path = os.path.join(self.images_dir, self.image_files[idx])
        label_path = os.path.join(self.labels_dir, os.path.splitext(self.image_files[idx])[0] + '.txt')

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image.shape

        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, cx, cy, bw, bh = map(float, parts)
                    x1 = (cx - bw / 2) * w
                    y1 = (cy - bh / 2) * h
                    x2 = (cx + bw / 2) * w
                    y2 = (cy + bh / 2) * h
                    boxes.append([x1, y1, x2, y2])
                    labels.append(1)

        if not boxes:
            boxes = [[0, 0, 1, 1]]
            labels = [0]

        if self.transform:
            augmented = self.transform(image=image, bboxes=boxes, labels=labels)
            image = augmented['image']
            boxes = augmented['bboxes']
            labels = augmented['labels']

        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64)
        }
        return image, target

    def __len__(self):
        return len(self.image_files)

# ---------- Swin Backbone ----------
class SwinBackbone(nn.Module):
    def __init__(self, out_channels=256):
        super().__init__()
        self.body = create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True)
        self.fpn_input_channels = [96, 192, 384, 768]
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_c, out_channels, kernel_size=1) for in_c in self.fpn_input_channels
        ])

    def forward(self, x):
        feats = self.body(x)
        fpn_feats = {}
        for idx, feat in enumerate(feats):
            if feat.dim() == 4 and feat.shape[-1] == self.fpn_input_channels[idx]:
                feat = feat.permute(0, 3, 1, 2)
            fpn_feats[str(idx)] = self.lateral_convs[idx](feat)
        return fpn_feats

# ---------- Identity Transform ----------
class IdentityTransform(nn.Module):
    def forward(self, images, targets=None):
        batch_images = torch.stack(images) if isinstance(images, list) else images
        image_sizes = [(224, 224) for _ in range(batch_images.shape[0])]
        return ImageList(batch_images, image_sizes), targets

    def postprocess(self, result, image_shapes, original_image_sizes):
        return result

# ---------- Build Model ----------
def get_swin_fasterrcnn(num_classes=2):
    backbone = SwinBackbone(out_channels=256)
    anchor_gen = AnchorGenerator(
        sizes=((16, 32), (32, 64), (64, 128), (128, 256)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 4
    )
    roi_pooler = MultiScaleRoIAlign(["0", "1", "2", "3"], output_size=7, sampling_ratio=2)
    model = FasterRCNN(backbone, num_classes, anchor_generator=anchor_gen, box_roi_pool=roi_pooler, transform=None)
    model.transform = IdentityTransform()
    return model

# ---------- Preprocessing ----------
transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# ---------- Data Loaders ----------
train_dataset = FireDetectionDataset("Satellite/train/images", "Satellite/train/labels", transform)
val_dataset = FireDetectionDataset("Satellite/valid/images", "Satellite/valid/labels", transform)
test_dataset = FireDetectionDataset("Satellite/test/images", "Satellite/test/labels", transform)

def collate_fn(batch): return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

# ---------- Train ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = get_swin_fasterrcnn(num_classes=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

print("🔧 Starting training...")
num_epochs = 10
model.train()
for epoch in range(num_epochs):
    total_loss, successful_batches = 0, 0
    for batch_idx, (images, targets) in enumerate(tqdm(train_loader)):
        try:
            images = torch.stack(images).to(device)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item(); successful_batches += 1
        except Exception as e:
            print(f"⚠️ Batch {batch_idx} error: {e}")
    avg_loss = total_loss / max(1, successful_batches)
    print(f"📘 Epoch {epoch+1}: Loss = {avg_loss:.4f}")

# ---------- Save ----------
torch.save(model.state_dict(), "swin_frcnn_satellite.pt")
print("✅ Model saved as swin_frcnn_satellite.pt")



##  Inference on Satellite Test Images using Trained YOLOv8 Model

We perform inference using the **best trained YOLOv8 model** to evaluate its performance on the **test set** of satellite fire scenarios.

- 🎯 **Model Weights**: `best.pt` from the training run
- 💾 **Output**: Predictions will be saved automatically by YOLO

This step visually validates how well the model generalizes to unseen indoor fire detection cases.


In [ ]:
# ---------- Test + Metrics ----------
model.eval()
TP = FP = FN = 0
ious = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="🚀 Testing"):
        images = torch.stack(images).to(device)
        outputs = model(images)
        pred_boxes = outputs[0]['boxes'].cpu()
        scores = outputs[0]['scores'].cpu()
        gt_boxes = targets[0]['boxes'].cpu()
        keep = scores >= 0.5
        pred_boxes = pred_boxes[keep]

        if len(pred_boxes) > 0 and len(gt_boxes) > 0:
            iou = box_iou(pred_boxes, gt_boxes)
            ious.extend(iou.max(dim=1).values.numpy().tolist())
            for i in range(len(pred_boxes)):
                if iou[i].max().item() >= 0.5:
                    TP += 1
                else:
                    FP += 1
            FN += len(gt_boxes) - TP
        elif len(gt_boxes) > 0:
            FN += len(gt_boxes)
        elif len(pred_boxes) > 0:
            FP += len(pred_boxes)

precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
f1 = 2 * (precision * recall) / (precision + recall + 1e-6)
avg_iou = np.mean(ious) if ious else 0.0

print(f"\n📊 Final Metrics:")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"Avg IoU:   {avg_iou:.4f}")

# ---------- Optional Visualization ----------
def visualize(image, boxes, title="Prediction"):
    img = image.permute(1, 2, 0).cpu().numpy()
    img = (img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.imshow(img)
    for box in boxes:
        x1, y1, x2, y2 = box
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1,
                     fill=False, color='lime', linewidth=2))
    ax.set_title(title); ax.axis("off"); plt.show()

# Visualize one result
sample_img, _ = test_dataset[0]
model.eval()
with torch.no_grad():
    out = model([sample_img.to(device)])
    visualize(sample_img.cpu(), out[0]['boxes'].cpu().detach())
